# Shopper Spectrum: Product Recommendation System

## Day 3 - Item Based Collaborative Filtering

### Objective
To recommend similar products using customer purchase history and cosine similarity.

# 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

import joblib

# 2. Load Dataset and Perform Cleaning

In [2]:
df = pd.read_csv(
    "../data/online_retail.csv",
    encoding="ISO-8859-1"
)

df = df.dropna(subset=["CustomerID"])

df = df[
    ~df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
]

df = df[df["Quantity"] > 0]

df = df[df["UnitPrice"] > 0]

print(df.shape)

(397884, 8)


# 3. Create Customer Product Matrix

In [9]:
top_products = (
    df.groupby("Description")["Quantity"]
      .sum()
      .sort_values(ascending=False)
      .head(1000)
      .index
)

df_filtered = df[
    df["Description"].isin(top_products)
]

customer_product_matrix = pd.pivot_table(
    df_filtered,
    index="CustomerID",
    columns="Description",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

customer_product_matrix.head()

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,...,WRAP VINTAGE LEAF DESIGN,WRAP VINTAGE PETALS DESIGN,WRAP WEDDING DAY,"WRAP, BILLBOARD FONTS DESIGN",YOU'RE CONFUSING ME METAL SIGN,ZINC FOLKART SLEIGH BELLS,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
CustomerID,,,,,,,,,,,,,,,,,,,,,
12346.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12347.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12348.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12349.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,12,0,0,0,0
12350.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 4. Transpose Matrix for Item-Based Filtering

In [15]:
product_matrix = customer_product_matrix.T

product_matrix.shape

(1000, 4286)

# 5. Calculate Cosine Similarity

In [16]:
similarity_matrix = cosine_similarity(
    product_matrix
)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=product_matrix.index,
    columns=product_matrix.index
)

similarity_df.iloc[:5, :5]

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,10 COLOUR SPACEBOY PEN
Description,,,,,
50'S CHRISTMAS GIFT BAG LARGE,1.000000,0.003533,0.900849,0.119031,0.021664
DOLLY GIRL BEAKER,0.003533,1.000000,0.003034,0.001765,0.841087
RED SPOT GIFT BAG LARGE,0.900849,0.003034,1.000000,0.117924,0.019649
SET 2 TEA TOWELS I LOVE LONDON,0.119031,0.001765,0.117924,1.000000,0.008905
10 COLOUR SPACEBOY PEN,0.021664,0.841087,0.019649,0.008905,1.000000


# 6. Recommendation Function

In [17]:
def recommend_products(
    product_name,
    n=5
):

    if product_name not in similarity_df.index:

        return [
            "Product Not Found"
        ]

    similar_products = (
        similarity_df[product_name]
        .sort_values(
            ascending=False
        )
        [1:n+1]
    )

    return similar_products.index.tolist()

# 7. Test Recommendation System

In [18]:
sample_product = similarity_df.index[0]

print(
    "Selected Product:",
    sample_product
)

recommend_products(
    sample_product
)

Selected Product:  50'S CHRISTMAS GIFT BAG LARGE


[' RED SPOT GIFT BAG LARGE',
 'RED RETROSPOT MUG',
 "50'S CHRISTMAS PAPER GIFT BAG",
 'RED RETROSPOT SUGAR JAM BOWL',
 'RED RETROSPOT SMALL MILK JUG']

# 8. Save Recommendation Models

In [20]:
joblib.dump(
    similarity_df,
    "../models/similarity.pkl"
)

joblib.dump(
    list(similarity_df.index),
    "../models/product_index.pkl"
)

print(
    "Recommendation Models Saved"
)

Recommendation Models Saved


### Observation

- Products were represented using customer purchase history.
- Item-based collaborative filtering was used to identify similar products.
- Cosine similarity measured product similarity.
- The recommendation system can suggest the top 5 most similar products.
- The saved model will be integrated into the Streamlit application.